# Week 1 — Linear Regression 1
### Integrated Capstone Project · Credit Risk Dataset
**Author: Gueorgui Poklitar**

This notebook is the foundation of the capstone modeling arc. The goal is to predict
**`loan_int_rate`** (the interest rate, in percentage points, assigned to each loan)
from borrower and loan attributes, and in doing so to exercise the core machinery of
ordinary least squares: *categorical and continuous features*, *multicollinearity and
the variance inflation factor*, *polynomial terms*, and *interaction terms*.

Week 2 builds directly on this notebook by adding regularization (Ridge, Lasso, Elastic
Net) on top of the same feature pipeline; everything established here — the cleaning
logic, the OLS baseline, the VIF diagnostics — is reused there. Treat this as Part 1 of
a two-part linear-regression story.

*Dataset: `laotse/credit-risk-dataset` (Kaggle), ~32k consumer loans.*

In [ ]:
#pip install pandas numpy scikit-learn matplotlib seaborn statsmodels kagglehub

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import kagglehub

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, root_mean_squared_error

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

print("All imports successful.")

## 1. Data Loading & Cleaning

In [ ]:
#Load from Kaggle
path = kagglehub.dataset_download("laotse/credit-risk-dataset")
df = pd.read_csv(f"{path}/credit_risk_dataset.csv")

print(f"Raw shape: {df.shape}")

#Cleaning
df = df.drop_duplicates()

# Median imputation for employment length (right-skewed, so median > mean here)
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())

# Grade-matched median for interest rate - preserves credit-grade logic
df['loan_int_rate'] = df['loan_int_rate'].fillna(
    df.groupby('loan_grade')['loan_int_rate'].transform('median')
)

# Remove logical impossibilities (not just statistical outliers)
df = df[df['person_emp_length'] <= df['person_age']]
df = df[df['person_age'] <= 100]
df = df[df['person_emp_length'] <= 60]

print(f"Clean shape: {df.shape}")
print(f"Default rate (loan_status=1): {df['loan_status'].mean()*100:.2f}%")
print(f"\nColumn types:\n{df.dtypes}")

---
## 2. Exploratory Baseline: Correlation & Feature Context

In [ ]:
numeric_cols = ['person_age', 'person_income', 'person_emp_length',
                'loan_amnt', 'loan_int_rate', 'loan_status',
                'loan_percent_income', 'cb_person_cred_hist_length']

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap - Credit Risk Features', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

print("Key correlations with loan_int_rate:")
print(corr['loan_int_rate'].sort_values(ascending=False).to_string())
print()
print("Interpretation: loan_int_rate correlates most strongly with loan_percent_income")
print("and loan_amnt among the numeric features. The strongest single driver - loan_grade -")
print("is categorical and does not appear here; it enters the model in Section 3.")

## 3. Categorical & Continuous Features: Baseline OLS with Full Pipeline

In [ ]:
# Define features
categorical_features = ['loan_intent', 'loan_grade',
                         'person_home_ownership', 'cb_person_default_on_file']
numeric_features = ['person_income', 'person_age', 'person_emp_length',
                    'loan_amnt', 'loan_percent_income']

X = df[categorical_features + numeric_features].copy()
y = df['loan_int_rate'].copy()

mask = X.notna().all(axis=1)
X, y = X[mask], y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Build preprocessing pipeline: scale numerics, one-hot encode categoricals
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

ols_pipeline = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', LinearRegression())
])

ols_pipeline.fit(X_train, y_train)
preds_ols = ols_pipeline.predict(X_test)

r2_ols   = r2_score(y_test, preds_ols)
rmse_ols = root_mean_squared_error(y_test, preds_ols)

print("Baseline OLS (categorical + continuous features)")
print(f"  Test R-squared : {r2_ols:.4f}")
print(f"  Test RMSE      : {rmse_ols:.4f} percentage points")
print()
print("Interpretation: A high R-squared here is driven almost entirely by loan_grade,")
print("which the lender assigns in lockstep with the rate. The continuous features")
print("(income, age, loan size) contribute the remaining, smaller slice of explained")
print("variance. Week 3 revisits this exact target WITHOUT loan_grade to force an honest model.")

## 4. Multicollinearity & Variance Inflation Factor (VIF)

In [ ]:
# VIF is computed on numeric-only features (no dummies for this diagnostic)
vif_features = ['person_age', 'person_income', 'person_emp_length',
                'loan_amnt', 'loan_percent_income',
                'cb_person_cred_hist_length', 'loan_int_rate']

df_vif = df[vif_features].dropna()
X_vif  = add_constant(df_vif)

vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF':     [variance_inflation_factor(X_vif.values, i)
                for i in range(X_vif.shape[1])]
}).query("Feature != 'const'").sort_values('VIF', ascending=False)

print(vif_data.to_string(index=False))
print()
print("Finding: person_age and cb_person_cred_hist_length show elevated VIF because")
print("they are nearly redundant (credit history length grows mechanically with age).")
print("In all downstream models, cb_person_cred_hist_length is dropped to avoid")
print("coefficient instability - this is the rule established here and reused in Weeks 2-3.")

#VIF Bar Chart
fig, ax = plt.subplots(figsize=(9, 5))
colors_vif = ['crimson' if v > 10 else 'orange' if v > 5 else 'steelblue'
              for v in vif_data['VIF']]
ax.barh(vif_data['Feature'], vif_data['VIF'], color=colors_vif)
ax.axvline(5,  color='orange', linestyle='--', linewidth=1.5, label='VIF=5 (caution)')
ax.axvline(10, color='crimson', linestyle='--', linewidth=1.5, label='VIF=10 (severe)')
ax.set_xlabel('VIF Score')
ax.set_title('Variance Inflation Factor - Credit Risk Numeric Features', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Polynomial Terms

In [ ]:
# Using the two most rate-relevant numeric features for a clean polynomial test
poly_features = ['loan_percent_income', 'loan_amnt']

df_poly = df[poly_features + ['loan_int_rate']].dropna()
X_poly_raw = df_poly[poly_features]
y_poly     = df_poly['loan_int_rate']

X_poly_train, X_poly_test, yp_train, yp_test = train_test_split(
    X_poly_raw, y_poly, test_size=0.2, random_state=42
)

results_poly = []
for deg in [1, 2, 3]:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ])
    pipe.fit(X_poly_train, yp_train)
    preds_p = pipe.predict(X_poly_test)
    results_poly.append({
        'Degree': deg,
        'R-squared': round(r2_score(yp_test, preds_p), 4),
        'RMSE': round(root_mean_squared_error(yp_test, preds_p), 4),
        'N Features': PolynomialFeatures(degree=deg).fit(X_poly_raw).n_output_features_
    })

poly_df = pd.DataFrame(results_poly)
print("Polynomial Regression - loan_percent_income & loan_amnt to loan_int_rate")
print(poly_df.to_string(index=False))

#Visual: scatter + polynomial fit (1D: loan_percent_income only)
df_1d = df[['loan_percent_income','loan_int_rate']].dropna()
df_1d = df_1d[df_1d['loan_percent_income'] < 0.8]
sample_1d = df_1d.sample(min(3000, len(df_1d)), random_state=42)

x_range = np.linspace(sample_1d['loan_percent_income'].min(),
                       sample_1d['loan_percent_income'].max(), 300).reshape(-1,1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(sample_1d['loan_percent_income'], sample_1d['loan_int_rate'],
           alpha=0.15, s=8, color='steelblue', label='Observations')

colors_deg = ['gray', 'darkorange', 'crimson']
for deg, col in zip([1, 2, 3], colors_deg):
    pipe_1d = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('model', LinearRegression())
    ])
    pipe_1d.fit(sample_1d[['loan_percent_income']], sample_1d['loan_int_rate'])
    ax.plot(x_range, pipe_1d.predict(x_range), color=col, linewidth=2.5,
            label=f'Degree {deg}')

ax.set_xlabel('loan_percent_income')
ax.set_ylabel('loan_int_rate')
ax.set_title('Polynomial Fits - Does Curvature Help?', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print()
print("Interpretation: The marginal R-squared gain from degree 2 or 3 is tiny. The")
print("rate-vs-debt-burden relationship is essentially linear; added curvature mostly")
print("fits noise. This justifies keeping linear terms in the main model and treating")
print("polynomial expansion as a tested-and-rejected hypothesis, not a default.")

## 6. Interaction Terms

In [ ]:
df_int = df[['person_income','person_emp_length','loan_int_rate']].dropna().copy()

# Manually create interaction term for transparency
df_int['income_x_emp'] = df_int['person_income'] * df_int['person_emp_length']

Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    df_int[['person_income','person_emp_length','income_x_emp']],
    df_int['loan_int_rate'], test_size=0.2, random_state=42
)

results_int = []
for label, feats in [('No Interaction', ['person_income','person_emp_length']),
                      ('With Interaction', ['person_income','person_emp_length','income_x_emp'])]:
    sc_i = StandardScaler()
    X_tr_i = sc_i.fit_transform(Xi_train[feats])
    X_te_i = sc_i.transform(Xi_test[feats])
    lr_i = LinearRegression().fit(X_tr_i, yi_train)
    preds_i = lr_i.predict(X_te_i)
    results_int.append({
        'Model': label,
        'R-squared': round(r2_score(yi_test, preds_i), 4),
        'RMSE': round(root_mean_squared_error(yi_test, preds_i), 4)
    })
    if label == 'With Interaction':
        print("Coefficients (standardized):")
        for name, c in zip(feats, lr_i.coef_):
            print(f"  {name:<30}: {c:.5f}")
        print(f"  {'Intercept':<30}: {lr_i.intercept_:.4f}")

print()
print(pd.DataFrame(results_int).to_string(index=False))
print()
print("Interpretation: If income_x_emp carried a meaningful coefficient, the marginal")
print("effect of income on rate would depend on employment stability. A small interaction")
print("coefficient and a flat R-squared confirm that income and employment act mostly")
print("independently for rate-setting - so the simpler additive model is preferred.")

---
## 7. Bringing It Together: OLS with Polynomial + Interaction + Categorical

In [ ]:
# A single pipeline that combines everything from Sections 3-6:
#   - scaled continuous features WITH degree-2 polynomial + interaction expansion
#   - one-hot encoded categoricals
#   - cb_person_cred_hist_length deliberately excluded (VIF rule from Section 4)
poly_num = ['person_income', 'person_age', 'person_emp_length',
            'loan_amnt', 'loan_percent_income']
cat_feats = ['loan_intent', 'loan_grade', 'person_home_ownership', 'cb_person_default_on_file']

X = df[poly_num + cat_feats].copy()
y = df['loan_int_rate'].copy()
mask = X.notna().all(axis=1)
X, y = X[mask], y[mask]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_poly_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)),
    ('scale', StandardScaler())
])
full_prep = ColumnTransformer([
    ('numpoly', num_poly_pipe, poly_num),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_feats)
])

models = {
    'Simple OLS (linear, no poly)': Pipeline([
        ('prep', ColumnTransformer([
            ('num', StandardScaler(), poly_num),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_feats)])),
        ('lr', LinearRegression())]),
    'Full OLS (+ poly2 + interactions)': Pipeline([
        ('prep', full_prep), ('lr', LinearRegression())]),
}

rows = []
for name, m in models.items():
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    cv = cross_val_score(m, X_train, y_train, cv=5, scoring='r2')
    rows.append({'Model': name,
                 'Test R-squared': round(r2_score(y_test, p), 4),
                 'Test RMSE': round(root_mean_squared_error(y_test, p), 4),
                 'CV R-squared Mean': round(cv.mean(), 4),
                 'CV R-squared Std': round(cv.std(), 4)})

compare = pd.DataFrame(rows)
print(compare.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(['Simple OLS', 'Full OLS\n(+poly2 +interactions)'], compare['Test R-squared'],
       color=['#555555', 'steelblue'], edgecolor='white')
for i, v in enumerate(compare['Test R-squared']):
    ax.text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Test R-squared')
ax.set_title('Does Polynomial + Interaction Expansion Earn Its Complexity?', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print()
print("What this means for the Chief Risk Officer: the elaborate feature expansion buys")
print("almost nothing over the simple linear model. The honest recommendation is to keep")
print("the parsimonious specification - it is easier to audit, explain to regulators, and")
print("less prone to overfitting. Complexity must justify itself; here it does not.")

### Week 1 Takeaways and the Bridge to Week 2

**What this week established:**
- A reusable cleaning recipe (median imputation, grade-matched rate imputation, removal of
  logical impossibilities) that every later notebook inherits unchanged.
- The OLS baseline for predicting `loan_int_rate`, and the finding that `loan_grade`
  dominates explained variance because the lender assigns rate *from* grade.
- A multicollinearity rule: `cb_person_cred_hist_length` is dropped because it is nearly
  collinear with `person_age` (high VIF).
- Tested-and-rejected complexity: neither polynomial terms nor an income x employment
  interaction meaningfully improve fit. The data want a simple, additive linear model.

**The bridge to Week 2:** Week 1 leaves us with a model whose coefficients are stable but
whose feature set is wide once categoricals are one-hot encoded (30+ columns). That is
exactly the regime where **regularization** earns its keep. Week 2 takes this identical
pipeline and adds Ridge, Lasso, and Elastic Net to ask: *which of these many features are
actually irreducible, and how much can we shrink the rest without hurting accuracy?*

**The bridge to Week 3:** The uncomfortable truth surfaced in Section 3 - that we are
partly predicting rate from a grade that was itself derived from rate - motivates Week 3,
where the same target is modeled **without `loan_grade`**, using forward/backward
selection, PCR, and PLSR to find an honest predictive signal in genuine borrower attributes.